# IBKR API notebook

#### Connection

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from ib_async import *
import pandas as pd
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)
if ib.isConnected():
    print("✅ Connected to IBKR API")
else:
    print("❌Failed to connect to IBKR API")
util.logToConsole(logging.INFO)

## Request Historical data

#### Choose your contract

In [ ]:
#contract = CFD('IBUST100', 'SMART', 'USD')
#contract = Forex(pair="EURUSD", exchange='IDEALPRO')
contract = Index('NDX', 'NASDAQ', 'USD')
contract_details = ib.reqContractDetails(contract)
# Extract and display the desired fields from contract_details
filtered_details = [
    {
        "secType": detail.contract.secType,
        "conId": detail.contract.conId,
        "symbol": detail.contract.symbol,
        "exchange": detail.contract.exchange,
        "longName": detail.longName,
        "timezoneId": detail.timeZoneId,
        "tradingHours": "\n".join(
            [f"  {segment}" for segment in detail.tradingHours.split(";")]
        ),
        "liquidHours": "\n".join(
            [f"  {segment}" for segment in detail.liquidHours.split(";")]
        ),
        "minSize": detail.minSize,
    }
    for detail in contract_details
]

# Print the filtered details in a clear format
for idx, detail in enumerate(filtered_details, start=1):
    print(f"Contract Detail {idx}:")
    for key, value in detail.items():
        print(f"  {key}: {value}")
    print()

#### Check first data timestamp available

In [ ]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=False)
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
logging.info(f"First date of data available: {formatted_time}")

### Request historical data function

**End date choice**

In [ ]:
save_path = "./database/AAPL_10secs_20220131_to_20250403.parquet"
# Load the parquet file into a DataFrame
retrieved_df = pd.read_parquet(save_path)
# Retrieve the first date from your existing DataFrame
first_date = retrieved_df.iloc[0]['date']  # Assuming 'date' is the column name
end_date = first_date.strftime('%Y%m%d %H:%M:%S')  # Format as 'yyyyMMdd HH:mm:ss'
logging.info(f"First date in the DataFrame: {first_date}")
logging.info(f"End date: {end_date}")

In [ ]:
# yesterday's date
end_date = (pd.Timestamp.now(tz='UTC') - pd.DateOffset(days=1)).strftime('%Y%m%d %H:%M:%S')

In [ ]:
#today's date
end_date = pd.Timestamp.now(tz='UTC').strftime('%Y%m%d-%H:%M:%S')

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`.  
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [ ]:
historical_data_interval = '10 secs' 
request_duration = '1 M'  # Duration in days (use D, not "day")
price_source = 'TRADES'  # 'BID', 'ASK', or 'TRADES'
bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow=price_source,
        useRTH=False, 
        formatDate=2,
        timeout = 0)

Convert the list of bars to a data frame and print the first and last rows:

In [ ]:
bars[0]
df = util.df(bars)

display(df.head())
display(df.tail())

#### Checking if volume and average columns are empty or not and remove it if empty

In [ ]:
# Remove the 'volume', 'average', and 'barCount' columns from the DataFrame
df = df.drop(columns=['volume', 'average', 'barCount'])

# Display the updated DataFrame
df.head()

#### Sauvegarde du fichier

Construction du nom du fichier et sauvegarde en `.parquet` dans le dossier `database`

Structure du nom du fichier : `Symbol_Interval_StartDate_EndDate_PriceSource.parquet`


In [ ]:
# Récupérer la devise et l'unité de temps
symbol = contract.symbol

# Construire le nom du fichier
start_date = df['date'].iloc[0].strftime('%Y%m%d')
end_date = df['date'].iloc[-1].strftime('%Y%m%d')
# Structure
save_path = f"../marketData/{symbol}_{historical_data_interval.replace(' ', '')}_{start_date}_to_{end_date}_{price_source}.parquet"

# Sauvegarder le DataFrame en fichier parquet
df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

#### Chargement d'un fichier

fichier csv

In [4]:
import pandas as pd
import os
save_path = "../marketData/NDX_30secs_20220214_to_20250502_TRADES.csv"

retrieved_df = pd.read_csv(save_path)
display(retrieved_df.head())

FileNotFoundError: [Errno 2] No such file or directory: '../marketData/NDX_30secs_20220214_to_20250502_TRADES.csv'

fichier parquet

In [8]:
import pandas as pd
import os
from igtrader.Strategies.Helpers import load_data, resample_ohlc
save_path = "../marketData/NDX_10secs_20220214_to_20250502_TRADES.parquet"
#save_path = "../marketData/NDX_10secs_20220131_to_20250403.parquet"

# Load the parquet file into a DataFrame

retrieved_df = pd.read_parquet(save_path)

# Display the first and last rows of the DataFrame
display(retrieved_df.head())
display(retrieved_df.tail())

,date,open,high,low,close
0,2022-02-14 14:30:10+00:00,14233.40,14239.47,14233.40,14239.47
1,2022-02-14 14:30:20+00:00,14239.47,14242.25,14231.47,14235.56
2,2022-02-14 14:30:30+00:00,14235.56,14242.16,14235.56,14239.24
3,2022-02-14 14:30:40+00:00,14239.24,14239.45,14233.44,14236.41
4,2022-02-14 14:30:50+00:00,14236.41,14257.50,14236.41,14257.50


,date,open,high,low,close
1897462,2025-05-02 19:59:10+00:00,20092.85,20099.47,20092.85,20099.47
1897463,2025-05-02 19:59:20+00:00,20100.19,20106.53,20100.19,20106.53
1897464,2025-05-02 19:59:30+00:00,20107.10,20110.80,20103.41,20110.80
1897465,2025-05-02 19:59:40+00:00,20111.43,20112.79,20103.82,20103.82
1897466,2025-05-02 19:59:50+00:00,20106.25,20106.56,20097.16,20098.05


In [9]:
retrieved_df = resample_ohlc(retrieved_df, '1min')
retrieved_df

,open,high,low,close
date,,,,
2022-02-14 14:31:00+00:00,14257.50,14267.11,14238.47,14238.47
2022-02-14 14:32:00+00:00,14238.47,14238.47,14209.07,14209.81
2022-02-14 14:33:00+00:00,14209.81,14214.50,14196.92,14198.70
2022-02-14 14:34:00+00:00,14198.70,14238.12,14198.41,14238.12
2022-02-14 14:35:00+00:00,14238.12,14260.28,14238.12,14242.30
...,...,...,...,...
2025-05-02 19:55:00+00:00,20106.89,20106.89,20080.33,20086.86
2025-05-02 19:56:00+00:00,20087.06,20095.04,20087.06,20092.08
2025-05-02 19:57:00+00:00,20095.48,20105.15,20095.48,20103.06


In [10]:
retrieved_df.to_csv("../marketData/NDX_1min_20220214_to_20250502_TRADES.csv", index=True)
import pandas as pd
import os
save_path = "../marketData/NDX_1min_20220214_to_20250502_TRADES.csv"

retrieved_df = pd.read_csv(save_path)
display(retrieved_df.head())

,date,open,high,low,close
0,2022-02-14 14:31:00+00:00,14257.50,14267.11,14238.47,14238.47
1,2022-02-14 14:32:00+00:00,14238.47,14238.47,14209.07,14209.81
2,2022-02-14 14:33:00+00:00,14209.81,14214.50,14196.92,14198.70
3,2022-02-14 14:34:00+00:00,14198.70,14238.12,14198.41,14238.12
4,2022-02-14 14:35:00+00:00,14238.12,14260.28,14238.12,14242.30


#### Conversion en UTC

In [ ]:
retrieved_df['date'] = retrieved_df['date'].dt.tz_convert('UTC')
retrieved_df

#### Enregistrement dataframe en UTC

In [ ]:
retrieved_df.to_parquet(save_path, index=True, compression=None)

## Additional features

#### Data pre processing

In [ ]:
import os
import pandas as pd
from igtrader.Strategies.Helpers import load_data, resample_ohlc

In [ ]:
resampled_df = '../marketData/IBUST100_10secs_20240922_to_20250422_ASK.parquet'
save_path = '../marketData/IBUST100_30secs_20240922_to_20250422_ASK.parquet'
resampled_df = pd.read_parquet(resampled_df)
resampled_df = resample_ohlc(resampled_df, '30secs')
resampled_df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {resampled_df}")
resampled_df


#### DataFrame jointure

In [ ]:
import pandas as pd

symbol = 'NDX'
interval = '10secs'
start_date = '20220214'
end_date = '20250502'
price_source= 'TRADES'

# 1. Load the existing dataframe
existing_file_path = "../marketData/NDX_10secs_20220214_to_20250411_TRADES.parquet"
existing_df = pd.read_parquet(existing_file_path)

# 3. Check for the last timestamp in existing data
last_timestamp = existing_df['date'].max()
print(f"Last timestamp in existing data: {last_timestamp}")

updated_df_path = "../marketData/NDX_10secs_20250403_to_20250502_TRADES.parquet"
updated_df = pd.read_parquet(updated_df_path)

# 5. Concatenate the dataframes
merged_df = pd.concat([existing_df, updated_df])

# 6. Sort the merged dataframe by date
merged_df = merged_df.sort_values('date')

# 7. Reset the index to create a clean sequential index
merged_df = merged_df.reset_index(drop=True)

# Display summary
print(f"Original data points: {len(existing_df)}")
print(f"New data points: {len(updated_df)}")
print(f"Total data points after merge: {len(merged_df)}")
merged_df

In [ ]:
save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}_{price_source}.parquet"
merged_df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

In [ ]:
from ib_async import *
util.startLoop()
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)

#contract = CFD('IBUST100', 'SMART', 'USD')
contract = Forex('EURUSD')
#contract = Index('NDX', 'NASDAQ', 'USD')

bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='500 S',
        barSizeSetting='5 secs',
        whatToShow='MIDPOINT',
        useRTH=False,
        formatDate=2,
        keepUpToDate=True)


def onBarUpdate(bars, hasNewBar):
    plt.close()
    plot = util.barplot(bars)
    plot.set_size_inches(18, 10)  # Width: 15 inches, Height: 8 inches
    clear_output(wait=True)
    display(plot)

bars.updateEvent += onBarUpdate

ib.sleep(60)
ib.cancelHistoricalData(bars)
ib.disconnect()